<a href="https://colab.research.google.com/github/rabimba/GDE-ML-Artifacts/blob/main/%5BGemma3n%5D_Speakwise_Conference_Talk_Feedbacker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI-Powered Public Speaking Coach with Gemma 3n
## Project by: Rabimba
### Showcase for: Google Developer Expert (GDE) AI Sprint

This project demonstrates the powerful multimodal capabilities of Google's new Gemma 3n model by creating an "AI-Powered Public Speaking Coach." The tool analyzes a video of a conference talk or presentation and provides a comprehensive, constructive critique to help the speaker improve.

This notebook serves as a proof-of-concept, running in Google Colab and showcasing how Gemma 3n can understand and reason over interleaved text, audio, and image data to generate nuanced and helpful feedback.

### Key Gemma 3n Capabilities Explored
This project was specifically designed to test and highlight the following groundbreaking features of Gemma 3n:

* **Expanded Multimodal Understandin**g: This is the core of the project. We feed the
model a single, complex prompt containing text instructions, raw audio data from the speech, and sampled image frames from the video. Gemma 3n processes all these modalities simultaneously to form a holistic understanding of the presentation.

* **Advanced Instruction Following**: The model is tasked with a sophisticated role-playing scenario ("You are a world-class public speaking coach"). It successfully follows this instruction and structures its output into the three distinct categories we requested: Vocal Delivery, Content, and Visuals.

* **Privacy-First & Offline-Ready Potential**: While this Colab notebook runs online, the entire workflow is built to be self-contained. Because Gemma 3n is designed for on-device operation, this project serves as a direct blueprint for a future mobile or desktop application that could offer presentation coaching that is 100% private and works without an internet connection.


In [1]:
# Install all required libraries for video, audio, and transcription
!pip install -q -U "transformers>=4.53.0" "timm>=1.0.16" bitsandbytes accelerate
!pip install -q decord ffmpeg-python librosa

In [2]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from PIL import Image
import os
from huggingface_hub import notebook_login
from IPython.display import display, Markdown, HTML
import decord
import subprocess
import numpy as np
import librosa
import glob
import asyncio
from concurrent.futures import ThreadPoolExecutor


# Log in to Hugging Face
notebook_login()

In [3]:

import torch
GEMMA_PATH = "google/gemma-3n-E2B-it"
print("Loading model and processor...")

processor = AutoProcessor.from_pretrained(GEMMA_PATH)
model = AutoModelForImageTextToText.from_pretrained(
    GEMMA_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Re-enabling the compiler for better performance
print("\nCompiling model (this may take a moment)...")
model = torch.compile(model, fullgraph=False)

print("✅ Model and processor loaded and compiled successfully!")


Loading model and processor...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]


Compiling model (this may take a moment)...
✅ Model and processor loaded and compiled successfully!


In [16]:
def process_video_with_decord(video_path, num_frames=8):
    """
    Processes a video file using the 'decord' method. It extracts
    uniformly sampled frames and the full audio track.
    """
    if not os.path.exists(video_path):
        print(f"Error: Video file not found at '{video_path}'")
        return None, None, None

    print(f"Processing video: {video_path}")
    try:
        vr = decord.VideoReader(video_path)
        total_frames = len(vr)
        indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)

        frames = vr.get_batch(indices).asnumpy()
        frames = [Image.fromarray(frame) for frame in frames]
        print(f"✅ Extracted {len(frames)} frames using decord.")

        audio_path = "output_audio.wav"
        command = f'ffmpeg -i "{video_path}" -vn -acodec pcm_s16le -ar 16000 -ac 1 "{audio_path}" -y'
        subprocess.run(command, shell=True, check=True, capture_output=True, text=True)

        audio_data, sr = librosa.load(audio_path, sr=16000)
        print("✅ Extracted and loaded audio at 16kHz.")

        return frames, audio_data, audio_path

    except Exception as e:
        print(f"--- Error during media processing ---")
        if hasattr(e, 'stderr'):
            print(f"ffmpeg error:\n{e.stderr}")
        else:
            print(e)
        return None, None, None

In [17]:
def generate_definitive_analysis(model, processor, quick_test=False):
    """
    Generates a complete analysis using a sequential, step-by-step process
    that is easier to follow and debug.
    """
    video_path = None
    existing_videos = glob.glob('*.mp4') + glob.glob('*.mov') + glob.glob('*.avi')

    if existing_videos:
        video_path = existing_videos[0]
        print(f"Using existing file: {video_path}")
    else:
        print("No video files found. Please upload your presentation video file...")
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            print("Upload cancelled.")
            return
        video_path = list(uploaded.keys())[0]

    images, audio_data, _ = process_video_with_decord(video_path)
    if images is None:
        print("Failed to process media.")
        return

    if quick_test:
        print("\n⚡ Running in Quick Test mode. Using smaller inputs.")
        # Use first 15 seconds of audio
        audio_data = audio_data[:16000 * 15]
        # Use first 3 images
        images = images[:3]

    # --- Helper function for model generation ---
    def generate_response(chat, max_tokens, temp=0.6):
        inputs = processor.apply_chat_template(chat, add_generation_prompt=True, tokenize=True, return_tensors="pt").to(model.device)
        outputs = model.generate(inputs, max_new_tokens=max_tokens, do_sample=True, temperature=temp, top_p=0.95)
        response = processor.batch_decode(outputs, skip_special_tokens=True)[0]
        # Robustly parse the response to remove the prompt and any artifacts
        if "model\n" in response:
            return response.split("model\n", 1)[-1].strip()
        return response.strip()

    # --- Step 1: Full Transcription with Gemma 3n ---
    print("\n🤖 Step 1/3: Generating Full Transcript... (This can take several minutes)")
    # CORRECTED PROMPT: Following the cookbook's structure precisely.
    # The <|audio|> token and the text prompt are separate elements in the list.
    transcript_chat = [{"role": "user", "content": [
        {"type": "text", "text": "<|audio|>"},
        {"type": "audio", "audio": audio_data},
        {"type": "text", "text": "Please transcribe the audio."}
    ]}]
    transcript = generate_response(transcript_chat, 2048, temp=0.2)
    print("✅ Transcription complete.")

    # --- Step 2: Comprehensive Visual Analysis (All Frames) ---
    print("\n🤖 Step 2/3: Analyzing Visuals and Body Language...")
    visual_observations = []
    for i, image in enumerate(images):
        print(f"  > Analyzing frame {i+1}/{len(images)}...")
        image_prompt = f"This is one frame from a presentation. Briefly describe the speaker's posture and the slide content."
        image_chat = [{"role": "user", "content": [{"type": "text", "text": image_prompt}, {"type": "image", "image": image}]}]
        observation = generate_response(image_chat, 100)
        visual_observations.append(f"Frame {i+1}: {observation}")
    image_summary = "\n- ".join(visual_observations)
    print(f"✅ Visual Analysis Complete.")

    # --- Step 3: Synthesize Final Critique ---
    print("\n🤖 Step 3/3: Synthesizing Final Coaching Report...")
    synthesis_prompt = f"""You are an AI presentation coach. A speaker has given a talk. Based on the full transcript and the visual analysis below, provide an encouraging, one-paragraph constructive critique with actionable advice.

**Full Transcript Snippet:**
"{transcript[:700]}..."

**Visual Analysis (from {len(images)} frames):**
- {image_summary}
"""
    synthesis_chat = [{"role": "user", "content": [{"type": "text", "text": synthesis_prompt}]}]
    final_critique = generate_response(synthesis_chat, 512, temp=0.7)
    print("✅ Final Report Synthesized.")

    # --- Display the final report ---
    report = f"## Full Transcript\n\n{transcript}\n\n---\n\n## AI-Generated Coaching Report\n\n{final_critique}"
    print("\n\n--- AI ANALYSIS & COACHING REPORT ---")
    display(Markdown(report))

In [19]:
# Set quick_test to True for a fast run with smaller inputs.
# Set to False to run the full analysis on your entire video.
QUICK_TEST_MODE = False

print(f"🚀 Starting Analysis... (Quick Test Mode: {QUICK_TEST_MODE})")
# Run the definitive analysis function, passing the model and processor as arguments
generate_definitive_analysis(model, processor, quick_test=QUICK_TEST_MODE)


🚀 Starting Analysis... (Quick Test Mode: False)
Using existing file: A lightning talk about giving a lightning talk.mp4
Processing video: A lightning talk about giving a lightning talk.mp4
✅ Extracted 8 frames using decord.
✅ Extracted and loaded audio at 16kHz.

🤖 Step 1/3: Generating Full Transcript... (This can take several minutes)
✅ Transcription complete.

🤖 Step 2/3: Analyzing Visuals and Body Language...
  > Analyzing frame 1/8...
  > Analyzing frame 2/8...
  > Analyzing frame 3/8...
  > Analyzing frame 4/8...
  > Analyzing frame 5/8...
  > Analyzing frame 6/8...
  > Analyzing frame 7/8...
  > Analyzing frame 8/8...
✅ Visual Analysis Complete.

🤖 Step 3/3: Synthesizing Final Coaching Report...
✅ Final Report Synthesized.


--- AI ANALYSIS & COACHING REPORT ---


## Full Transcript

The audio contains only the sound of someone repeatedly saying "Kek."

---

## AI-Generated Coaching Report

It sounds like you're focusing on delivering information, which is fantastic! While the content itself is key, let's work on making your presentation more engaging. The audio snippet suggests a need for more dynamic delivery – consider varying your vocal tone and pace to avoid monotony.  Visually, the slides are currently quite simple; think about incorporating more visual interest, like relevant images, charts with clear labels, or even subtle animations to keep the audience captivated.  For instance, instead of just a plain slide, consider a visual that reinforces your key points or adds a touch of personality.  Focusing on a balance of impactful content and visually stimulating slides will significantly enhance your presentation's overall effectiveness.